# EIB: Six-Country Static Persona Baseline (c3110)

**Task:** with an LLM, predict each respondent's answer to **c3110** (expected change in household
finances over the next 12 months vs now, 1 = much worse … 5 = much better) from **static persona
characteristics only**, for Belgium, Germany, Spain, France, Italy and the Netherlands.

**Inputs (one set):**
- `six_country_static_persona_baseline_2020_2026.csv`: the persona file (what the LLM may see)
- the c3110 file from the same set: **the answers** (never shown to the LLM, only used for scoring)

The two join one-to-one on `a0010` (respondent ID) + `a0030` (wave), and the notebook checks that country
and month agree on both sides before anything is used.

**Out of scope here:** macro context and respondent history (added later by teammates). To make that easy,
the evaluation sample is saved to Drive and every row carries how many earlier c3110 answers the person has
(`n_prior`), so later models can be scored on exactly the same rows and the same 3 / 6 / 12-history subsets.

**Before you start:** add `ANTHROPIC_API_KEY` in Colab's 🔑 Secrets panel (toggle *Notebook access* on).
Run top to bottom; Step 6 is a cheap 12-row test, Step 7 the full run.

## 0. Settings

In [5]:
DATA_DIR = '/content/drive/MyDrive/EIB_ECB2'      # <- folder holding the files below
STATIC_FILE = 'six_country_static_persona_baseline_2020_2026.csv'
TARGET_FILE = 'six_country_c3110_labels_evaluation_only_2020_2026.csv'   # the answers; None = auto-detect *c3110*.csv
OUT_DIR = '/content/drive/MyDrive/EIB_runs/six_country_static_baseline'

COUNTRIES = ['BE', 'DE', 'ES', 'FR', 'IT', 'NL']
TARGET = 'c3110'
VALID = [1, 2, 3, 4, 5]
OPTIONS = {1: 'get a lot worse', 2: 'get a little worse', 3: 'stay about the same',
           4: 'get a little better', 5: 'get a lot better'}
QUESTION = ('Looking ahead to 12 months from now, do you expect the financial situation of your household '
            'to be better or worse than it is now?')

# Persona = the labelled static columns of the static file (country is always added)
PERSONA_FIELDS = {
    'gender': 'gender_released',
    'age group': 'age_group',
    'education': 'education_group',
    'household net income quintile (1 = lowest 20% in country, 5 = highest)': 'income_quintile',
    'household size': 'household_size_topcoded',
    'children aged 0-17 in household': 'children_0_17_topcoded',
    'lives with partner': 'partner_in_household',
}
# Not used: b2100_prec has no labelled version yet. Add it once its meaning is confirmed.

N_PER_COUNTRY = 200      # full run: 6 x 200 = 1,200 rows
N_TEST_PER_COUNTRY = 2   # test run: 12 rows
HISTORY_CUTS = [3, 6, 12]  # report results on rows whose person has at least this many earlier answers
SEED = 42

MODEL = 'claude-haiku-5-5'   # cheaper: 'claude-sonnet-5-5' or 'claude-haiku-5-5'
EFFORT = 'low'
MAX_CONCURRENCY = 8

## 1. Setup

In [6]:
import os, re, glob, json, time, asyncio
import numpy as np, pandas as pd
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    !pip -q install -U anthropic scikit-learn scipy
except ImportError:
    pass  # running outside Colab: set ANTHROPIC_API_KEY in the environment
os.makedirs(OUT_DIR, exist_ok=True)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Load the static persona file and check it

In [7]:
static = pd.read_csv(f'{DATA_DIR}/{STATIC_FILE}', low_memory=False)
print(static.shape)

problems = []
dup = static.duplicated(['a0010', 'a0030']).sum()
if dup: problems.append(f'{dup} duplicated (a0010, a0030) keys in static file')
bad_c = set(static.a0020) - set(COUNTRIES)
if bad_c: problems.append(f'unexpected countries in static file: {bad_c}')
missing_cols = [c for c in PERSONA_FIELDS.values() if c not in static.columns]
if missing_cols: problems.append(f'persona columns missing: {missing_cols}')
na = static[list(PERSONA_FIELDS.values())].isna().mean()
if (na > 0).any(): print('Missing share in persona fields:\n', na[na > 0])
leak = [c for c in static.columns if c.lower().startswith('c3') or c.lower().startswith('c1')]
if leak: problems.append(f'static file contains survey-answer columns {leak}: they must not reach the prompt')
assert not problems, '\n'.join(problems)

# nominal wave date rule from the CES guide; check the file agrees
exp_year, exp_month = 2020 + (static.a0030 - 1) // 12, (static.a0030 - 1) % 12 + 1
assert ((static.survey_year == exp_year) & (static.survey_month == exp_month)).all(), 'wave/month mismatch in static file'
print('Static file OK:', static.a0010.nunique(), 'people,', static.a0030.min(), '-', static.a0030.max(), 'waves')
print(static.a0020.value_counts())

(1089680, 34)
Static file OK: 114321 people, 4 - 81 waves
a0020
IT    239692
ES    231364
FR    231240
DE    226375
BE     81300
NL     79709
Name: count, dtype: int64


## 3. Load the c3110 answer file and join one-to-one

Checks: the key is unique on both sides, the join is one-to-one, country and month agree where the answer
file carries them, c3110 values are valid (1–5; anything else, e.g. -999 / -777 codes, is dropped), and
how many rows on each side did not match.

In [8]:
if TARGET_FILE is None:
    cands = [f for f in glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True) if 'c3110' in os.path.basename(f).lower()]
    print('c3110 file candidates:', cands)
    assert len(cands) == 1, 'Set TARGET_FILE to the c3110 answer file of this set'
    TARGET_FILE = os.path.basename(cands[0]); TARGET_PATH = cands[0]
else:
    TARGET_PATH = f'{DATA_DIR}/{TARGET_FILE}'

ans = pd.read_csv(TARGET_PATH, low_memory=False)
print(ans.shape, list(ans.columns))
if TARGET not in ans.columns and 'target_value' in ans.columns:   # long format: target_variable / target_value
    assert (ans['target_variable'] == TARGET).all(), f'answer file holds other variables: {ans.target_variable.unique()}'
    if 'target_valid' in ans.columns:
        print('rows flagged target_valid=False:', int((~ans.target_valid.astype(bool)).sum()))
        ans = ans[ans.target_valid.astype(bool)]
    ans = ans.rename(columns={'target_value': TARGET})
assert {'a0010', 'a0030', TARGET} <= set(ans.columns), f'answer file needs a0010, a0030 and {TARGET}'
assert not ans.duplicated(['a0010', 'a0030']).any(), 'duplicated (a0010, a0030) keys in answer file'

check_cols = [c for c in ['a0020', 'country', 'date', 'survey_year', 'survey_month', 'survey_period'] if c in ans.columns]
ans_small = ans[['a0010', 'a0030', TARGET] + check_cols].rename(columns={c: c + '_ans' for c in check_cols})
merged = static.merge(ans_small, on=['a0010', 'a0030'], how='outer', validate='one_to_one', indicator=True)
print('\nJoin result:'); print(merged['_merge'].value_counts().rename({'left_only': 'static only', 'right_only': 'answer only'}))
answer_only = merged[merged._merge == 'right_only']
if len(answer_only):
    print('answer-only rows by country:', answer_only.get('a0020_ans', pd.Series(dtype=str)).value_counts().to_dict())
df = merged[merged._merge == 'both'].drop(columns='_merge').copy()

for c in check_cols:
    left = df[c].astype(str); right = df[c + '_ans'].astype(str)
    n_bad = (left != right).sum()
    print(f'{c}: {n_bad} mismatches')
    assert n_bad == 0, f'{c} disagrees between the static and answer files for {n_bad} rows'
if not check_cols:
    print('Answer file has no country/month columns: only the key could be checked.')

df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')
invalid = (~df[TARGET].isin(VALID)).sum()
print(f'{TARGET}: {invalid} missing/invalid rows dropped ({invalid / len(df):.2%})')
df = df[df[TARGET].isin(VALID)].copy(); df[TARGET] = df[TARGET].astype(int)

# earlier valid answers per person (strictly earlier waves) -- only for subset reporting, never in the prompt
df = df.sort_values(['a0010', 'a0030'])
df['n_prior'] = df.groupby('a0010').cumcount()
print(f'\n{len(df):,} usable rows, {df.a0010.nunique():,} people')
print(pd.DataFrame({f'>= {k} prior': (df.n_prior >= k).groupby(df.a0020).mean() for k in HISTORY_CUTS}).round(3))
print('\nc3110 distribution by country (row shares):')
print(pd.crosstab(df.a0020, df[TARGET], normalize='index').round(3))

(1089680, 10) ['a0010', 'a0020', 'a0030', 'country', 'date', 'survey_period', 'target_variable', 'target_raw', 'target_value', 'target_valid']
rows flagged target_valid=False: 0

Join result:
_merge
both           1089680
static only          0
answer only          0
Name: count, dtype: int64
a0020: 0 mismatches
country: 0 mismatches
date: 0 mismatches
survey_period: 0 mismatches
c3110: 0 missing/invalid rows dropped (0.00%)

1,089,680 usable rows, 114,321 people
       >= 3 prior  >= 6 prior  >= 12 prior
a0020                                     
BE          0.725       0.586        0.376
DE          0.767       0.622        0.395
ES          0.778       0.631        0.396
FR          0.764       0.617        0.389
IT          0.797       0.652        0.414
NL          0.736       0.594        0.376

c3110 distribution by country (row shares):
c3110      1      2      3      4      5
a0020                                   
BE     0.117  0.249  0.517  0.099  0.018
DE     0.074  0.210 

## 4. Evaluation sample (fixed, saved, reused by later models)

Simple random sample of respondent-waves within each country, at most one row per person. Written once to
Drive; later runs re-load it so every model is scored on the same rows.

In [9]:
SAMPLE_PATH = f'{OUT_DIR}/eval_sample_seed{SEED}_n{N_PER_COUNTRY}.csv'
if os.path.exists(SAMPLE_PATH):
    keys = pd.read_csv(SAMPLE_PATH, usecols=['a0010', 'a0030'])
    sample = keys.merge(df, on=['a0010', 'a0030'], how='left', validate='one_to_one')
    assert sample[TARGET].notna().all(), 'saved sample does not match the data; delete it or change SEED'
    print('Re-loaded', SAMPLE_PATH)
else:
    sample = (df.sample(frac=1, random_state=SEED).drop_duplicates('a0010')
                .groupby('a0020', group_keys=False).head(N_PER_COUNTRY).reset_index(drop=True))
    sample.to_csv(SAMPLE_PATH, index=False); print('Saved', SAMPLE_PATH)
sample['row_id'] = sample.a0010.astype(str) + '_w' + sample.a0030.astype(str)
print(sample.groupby('a0020').size().to_dict(), '| waves', sample.a0030.min(), '-', sample.a0030.max())
print({f'>= {k} prior': int((sample.n_prior >= k).sum()) for k in HISTORY_CUTS})

Saved /content/drive/MyDrive/EIB_runs/six_country_static_baseline/eval_sample_seed42_n200.csv
{'BE': 200, 'DE': 200, 'ES': 200, 'FR': 200, 'IT': 200, 'NL': 200} | waves 4 - 81
{'>= 3 prior': 918, '>= 6 prior': 730, '>= 12 prior': 459}


## 5. Prompt

The prompt is built **only** from `PERSONA_FIELDS` + country. The cell asserts that neither the answer nor
any survey-answer column can appear in it, and prints one example so you can read it.

In [10]:
COUNTRY_NAMES = {'BE': 'Belgium', 'DE': 'Germany', 'ES': 'Spain', 'FR': 'France', 'IT': 'Italy', 'NL': 'the Netherlands'}
assert TARGET not in PERSONA_FIELDS.values()
assert not any(c.lower().startswith(('c1', 'c3')) for c in PERSONA_FIELDS.values())

SYSTEM = ('You simulate how a specific respondent of the ECB Consumer Expectations Survey answers. '
          'Predict what this particular person would answer, not what is economically correct. '
          'Give calibrated probabilities that reflect your real uncertainty about this person.')

def persona_text(row):
    lines = [f'- country: {COUNTRY_NAMES[row["a0020"]]}']
    for name, col in PERSONA_FIELDS.items():
        v = row[col]
        if isinstance(v, float) and v.is_integer(): v = int(v)
        if pd.notna(v): lines.append(f'- {name}: {str(v).replace(" (released recode)", "")}')
    return '\n'.join(lines)

def build_prompt(row):
    opts = '\n'.join(f'  {k}: {v}' for k, v in OPTIONS.items())
    return (f'Respondent profile:\n{persona_text(row)}\n\nQuestion: {QUESTION}\nOptions:\n{opts}\n\n'
            'Give the probability that this respondent picks each option (keys "1".."5", summing to 1).')

print(SYSTEM, '\n---\n' + build_prompt(sample.iloc[0]))

You simulate how a specific respondent of the ECB Consumer Expectations Survey answers. Predict what this particular person would answer, not what is economically correct. Give calibrated probabilities that reflect your real uncertainty about this person. 
---
Respondent profile:
- country: Italy
- gender: Male
- age group: 35-49
- education: Higher education
- household net income quintile (1 = lowest 20% in country, 5 = highest): 5
- household size: 4
- children aged 0-17 in household: 2
- lives with partner: Yes

Question: Looking ahead to 12 months from now, do you expect the financial situation of your household to be better or worse than it is now?
Options:
  1: get a lot worse
  2: get a little worse
  3: stay about the same
  4: get a little better
  5: get a lot better

Give the probability that this respondent picks each option (keys "1".."5", summing to 1).


## 6. LLM calls (structured JSON output, cached on Drive so a crash resumes)

In [11]:
import anthropic
client = anthropic.AsyncAnthropic()
SCHEMA = {'type': 'object',
          'properties': {'probs': {'type': 'object',
                                   'properties': {str(k): {'type': 'number'} for k in VALID},
                                   'required': [str(k) for k in VALID], 'additionalProperties': False}},
          'required': ['probs'], 'additionalProperties': False}

async def ask(row, sem):
    async with sem:
        for attempt in range(5):
            try:
                resp = await client.beta.messages.create(
                    model=MODEL, max_tokens=2000, system=SYSTEM,
                    messages=[{'role': 'user', 'content': build_prompt(row)}],
                    output_config={'effort': EFFORT, 'format': {'type': 'json_schema', 'schema': SCHEMA}},
                    betas=['server-side-fallback-2026-07-01'], fallbacks='default',
                )
                if resp.stop_reason == 'refusal':
                    return {'row_id': row['row_id'], 'error': 'refusal'}
                out = json.loads(next(b.text for b in resp.content if b.type == 'text'))
                return {'row_id': row['row_id'], 'probs': out['probs'], 'model': resp.model,
                        'in_tok': resp.usage.input_tokens, 'out_tok': resp.usage.output_tokens}
            except (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError):
                await asyncio.sleep(5 * 2 ** attempt)
            except Exception as e:
                return {'row_id': row['row_id'], 'error': repr(e)[:300]}
        return {'row_id': row['row_id'], 'error': 'retries exhausted'}

def cache_path(tag): return f'{OUT_DIR}/answers_{tag}_{MODEL}.jsonl'

def load_answers(path):
    recs = [json.loads(l) for l in open(path)]
    ok = {r['row_id']: r for r in recs if 'error' not in r}
    errs = {r['row_id']: r['error'] for r in recs if 'error' in r and r['row_id'] not in ok}
    if errs: print(f'{len(errs)} rows failed, e.g. {next(iter(errs.values()))} (re-run the cell to retry)')
    rows = []
    for r in ok.values():
        p = np.array([max(float(r['probs'][str(k)]), 0.0) for k in VALID])
        p = p / p.sum() if p.sum() > 0 else np.full(len(VALID), 1 / len(VALID))
        rows.append({'row_id': r['row_id'], **{f'p{k}': p[i] for i, k in enumerate(VALID)}, 'pred': VALID[int(p.argmax())]})
    return pd.DataFrame(rows)

async def run_rows(rows, tag):
    path = cache_path(tag)
    done = {json.loads(l)['row_id'] for l in open(path) if '"error"' not in l} if os.path.exists(path) else set()
    todo = [r for _, r in rows.iterrows() if r['row_id'] not in done]
    print(f'{len(done)} cached, {len(todo)} to call ({MODEL}, effort={EFFORT})')
    sem = asyncio.Semaphore(MAX_CONCURRENCY); t0 = time.time(); tok = [0, 0]
    tasks = [asyncio.create_task(ask(r, sem)) for r in todo]
    with open(path, 'a') as f:
        for i, fut in enumerate(asyncio.as_completed(tasks), 1):
            res = await fut; f.write(json.dumps(res) + '\n'); f.flush()
            tok[0] += res.get('in_tok', 0); tok[1] += res.get('out_tok', 0)
            if i % 100 == 0 or i == len(tasks): print(f'  {i}/{len(tasks)}  {time.time()-t0:.0f}s  tokens in/out {tok}')
    return load_answers(path)

### 6b. Test on 12 rows (check the output looks sensible before the full run)

In [12]:
test_rows = sample.groupby('a0020', group_keys=False).head(N_TEST_PER_COUNTRY)
test_ans = await run_rows(test_rows, 'test')
test_ans.merge(test_rows[['row_id', 'a0020', TARGET]], on='row_id').round(2)

0 cached, 12 to call (claude-haiku-5-5, effort=low)
  12/12  4s  tokens in/out [7584, 588]


,row_id,p1,p2,p3,p4,p5,pred,a0020,c3110
0,p_45697196_w14,0.10,0.27,0.42,0.16,0.05,3,DE,3
1,p_64822670_w58,0.07,0.25,0.42,0.20,0.06,3,IT,3
2,p_83153814_w37,0.08,0.27,0.36,0.24,0.05,3,FR,4
3,p_47248092_w33,0.10,0.30,0.42,0.13,0.05,3,ES,3
4,p_60619146_w76,0.07,0.23,0.46,0.20,0.04,3,DE,2
5,p_20599389_w39,0.05,0.20,0.42,0.28,0.05,3,FR,1
6,p_34943365_w41,0.10,0.30,0.36,0.20,0.04,3,IT,2
7,p_90180310_w57,0.07,0.25,0.42,0.20,0.06,3,NL,2
8,p_75321020_w61,0.10,0.27,0.42,0.14,0.07,3,ES,3
9,p_90034727_w21,0.07,0.25,0.46,0.18,0.04,3,NL,3


## 7. Full run

In [13]:
answers = await run_rows(sample, 'full')
print(answers.shape)

0 cached, 1200 to call (claude-haiku-5-5, effort=low)
  100/1200  15s  tokens in/out [63230, 4900]
  200/1200  29s  tokens in/out [126431, 9800]
  300/1200  44s  tokens in/out [189643, 14700]
  400/1200  59s  tokens in/out [252865, 19600]
  500/1200  72s  tokens in/out [316100, 24500]
  600/1200  87s  tokens in/out [379312, 29400]
  700/1200  100s  tokens in/out [442565, 34300]
  800/1200  115s  tokens in/out [505770, 39200]
  900/1200  129s  tokens in/out [568984, 44100]
  1000/1200  144s  tokens in/out [632319, 49000]
  1100/1200  158s  tokens in/out [695766, 53900]
  1200/1200  172s  tokens in/out [759171, 58800]
(1200, 7)


## 8. Reference baselines (no LLM)

All of them only use answers from **earlier waves** than the row being predicted (no look-ahead).
- **random**: uniform over 1–5.
- **permutation**: real sample answers shuffled within country (right distribution, no individual signal).
- **country_mode**: the country's answer shares in earlier waves; prediction = most common answer.
- **persona_cell**: same, within country × age group × income quintile (a simple non-LLM static persona
  model; falls back to country when the cell has < 30 earlier answers). The LLM should beat this one.

In [14]:
rng = np.random.default_rng(SEED)
K = len(VALID)

def shares(s):
    return s.value_counts(normalize=True).reindex(VALID, fill_value=0).values

def to_frame(row_ids, P):
    P = np.asarray(P, dtype=float)
    out = pd.DataFrame(P, columns=[f'p{k}' for k in VALID]); out.insert(0, 'row_id', list(row_ids))
    out['pred'] = np.array(VALID)[P.argmax(axis=1)]
    return out

def prior_shares(row, cols, min_n=30):
    mask = (df.a0030 < row['a0030'])
    for c in cols: mask &= (df[c] == row[c])
    past = df.loc[mask, TARGET]
    return shares(past) if len(past) >= min_n else None

def make_baselines(s):
    B = {'random': to_frame(s.row_id, np.full((len(s), K), 1 / K))}
    B['random']['pred'] = rng.choice(VALID, len(s))
    perm = s.groupby('a0020')[TARGET].transform(lambda x: x.sample(frac=1, random_state=SEED).values).values
    B['permutation'] = to_frame(s.row_id, np.eye(K)[perm - 1])
    cm, cell = [], []
    for _, r in s.iterrows():
        c = prior_shares(r, ['a0020'], 1)
        c = c if c is not None else shares(df.loc[df.a0020 == r['a0020'], TARGET])  # first wave
        cm.append(c)
        z = prior_shares(r, ['a0020', 'age_group', 'income_quintile'])
        cell.append(z if z is not None else c)
    B['country_mode'] = to_frame(s.row_id, cm)
    B['persona_cell'] = to_frame(s.row_id, cell)
    return B

BASELINES = make_baselines(sample)

## 9. Scores

Higher is better: accuracy, macro-F1. Lower is better: Brier (multiclass), JS divergence (predicted vs actual
answer distribution, per country, averaged). Diversity: entropy ratio and option coverage (1.0 = as spread out
as real answers; the earlier Sonnet run was ~0.37 entropy ratio, i.e. answers bunched together).

In [15]:
from sklearn.metrics import f1_score, accuracy_score
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy

def score(pred, s):
    m = s.merge(pred, on='row_id', how='inner')
    y, yhat = m[TARGET].values, m['pred'].values
    P = m[[f'p{k}' for k in VALID]].values; Y = np.eye(K)[y - 1]
    js = np.mean([jensenshannon(shares(g[TARGET]), shares(g['pred']), base=2) for _, g in m.groupby('a0020')])
    return {'n': len(m), 'accuracy': accuracy_score(y, yhat), 'f1_macro': f1_score(y, yhat, average='macro'),
            'brier': np.mean(np.sum((P - Y) ** 2, axis=1)), 'js_divergence': js,
            'entropy_ratio': entropy(shares(pd.Series(yhat))) / entropy(shares(pd.Series(y))),
            'option_coverage': len(set(yhat)) / len(set(y))}

ALL = {f'LLM {MODEL}': answers, **BASELINES}
subsets = {'all': sample, **{f'>= {k} prior': sample[sample.n_prior >= k] for k in HISTORY_CUTS}}
summary = pd.concat({sub: pd.DataFrame({name: score(p, s) for name, p in ALL.items()}).T
                     for sub, s in subsets.items()})
summary.round(3)

n  accuracy  f1_macro  brier  js_divergence  entropy_ratio  option_coverage
all         LLM claude-haiku-5-5  1200.0     0.522     0.141  0.646          0.526          0.032              0.4
            random                1200.0     0.193     0.149  0.800          0.382          1.300              1.0
            permutation           1200.0     0.367     0.210  1.267          0.000          1.000              1.0
            country_mode          1200.0     0.521     0.138  0.642          0.537          0.014              0.4
            persona_cell          1200.0     0.519     0.140  0.631          0.524          0.039              0.4
>= 3 prior  LLM claude-haiku-5-5   918.0     0.539     0.145  0.639          0.513          0.037              0.4
            random                 918.0     0.199     0.154  0.800          0.378          1.316              1.0
            permutation            918.0     0.380     0.216  1.240          0.052          1.001              1.0
            country_mode           918.0     0.538     0.140  0.630          0.530          0.000              0.2
            persona_cell           918.0     0.536     0.140  0.619          0.525          0.013              0.4
>= 6 prior  LLM claude-haiku-5-5   730.0     0.538     0.147  0.641          0.514          0.034              0.4
            random                 730.0     0.201     0.156  0.800          0.366          1.315              1.0
            permutation            730.0     0.388     0.219  1.225          0.052          0.991              1.0
            country_mode           730.0     0.536     0.140  0.633          0.532          0.000              0.2
            persona_cell           730.0     0.534     0.139  0.624          0.527          0.009              0.4
>= 12 prior LLM claude-haiku-5-5   459.0     0.553     0.153  0.637          0.502          0.041              0.4
            random                 459.0     0.187     0.146  0.800          0.378          1.328              1.0
            permutation            459.0     0.388     0.216  1.224          0.102          1.015              1.0
            country_mode           459.0     0.549     0.142  0.626          0.523          0.000              0.2
            persona_cell           459.0     0.549     0.142  0.615          0.523          0.000              0.2

In [16]:
per_country = pd.concat({c: pd.DataFrame({name: score(p, sample[sample.a0020 == c]) for name, p in ALL.items()}).T
                         for c in COUNTRIES})
per_country[['n', 'accuracy', 'f1_macro', 'brier', 'entropy_ratio']].round(3)

n  accuracy  f1_macro  brier  entropy_ratio
BE LLM claude-haiku-5-5  200.0     0.500     0.133  0.656          0.000
   random                200.0     0.225     0.162  0.800          1.301
   permutation           200.0     0.355     0.201  1.290          1.000
   country_mode          200.0     0.500     0.133  0.658          0.000
   persona_cell          200.0     0.500     0.134  0.640          0.026
DE LLM claude-haiku-5-5  200.0     0.590     0.148  0.617          0.000
   random                200.0     0.180     0.145  0.800          1.403
   permutation           200.0     0.450     0.252  1.100          1.000
   country_mode          200.0     0.590     0.148  0.592          0.000
   persona_cell          200.0     0.590     0.148  0.582          0.000
ES LLM claude-haiku-5-5  200.0     0.550     0.142  0.647          0.026
   random                200.0     0.190     0.138  0.800          1.313
   permutation           200.0     0.350     0.184  1.300          1.000
   country_mode          200.0     0.555     0.143  0.630          0.000
   persona_cell          200.0     0.555     0.143  0.627          0.000
FR LLM claude-haiku-5-5  200.0     0.490     0.139  0.675          0.024
   random                200.0     0.165     0.136  0.800          1.244
   permutation           200.0     0.310     0.178  1.380          1.000
   country_mode          200.0     0.485     0.131  0.678          0.000
   persona_cell          200.0     0.485     0.131  0.666          0.000
IT LLM claude-haiku-5-5  200.0     0.500     0.141  0.647          0.097
   random                200.0     0.185     0.139  0.800          1.324
   permutation           200.0     0.395     0.253  1.210          1.000
   country_mode          200.0     0.495     0.139  0.650          0.064
   persona_cell          200.0     0.485     0.137  0.638          0.097
NL LLM claude-haiku-5-5  200.0     0.505     0.140  0.635          0.026
   random                200.0     0.215     0.159  0.800          1.328
   permutation           200.0     0.340     0.177  1.320          1.000
   country_mode          200.0     0.500     0.133  0.645          0.000
   persona_cell          200.0     0.500     0.145  0.632          0.081

In [17]:
# Predicted vs actual answer distribution (all sampled rows)
dist = pd.DataFrame({'actual': shares(sample[TARGET]),
                     **{name: shares(sample[['row_id']].merge(p, on='row_id')['pred']) for name, p in ALL.items()}},
                    index=[f'{k}: {v}' for k, v in OPTIONS.items()])
dist.round(3)

,actual,LLM claude-haiku-5-5,random,permutation,country_mode,persona_cell
1: get a lot worse,0.085,0.000,0.209,0.085,0.000,0.000
2: get a little worse,0.246,0.007,0.167,0.246,0.002,0.008
3: stay about the same,0.522,0.993,0.219,0.522,0.998,0.992
4: get a little better,0.128,0.000,0.207,0.128,0.000,0.000
5: get a lot better,0.020,0.000,0.198,0.020,0.000,0.000


## 10. Save to Drive

In [18]:
stamp = time.strftime('%Y%m%d_%H%M')
out = f'{OUT_DIR}/run_{stamp}_{MODEL}'; os.makedirs(out, exist_ok=True)
summary.to_csv(f'{out}/summary.csv'); per_country.to_csv(f'{out}/per_country.csv'); dist.to_csv(f'{out}/distribution.csv')
preds = sample[['row_id', 'a0010', 'a0030', 'a0020', 'n_prior', TARGET]].merge(answers, on='row_id', how='left')
preds.to_csv(f'{out}/llm_predictions.csv', index=False)
json.dump({'model': MODEL, 'effort': EFFORT, 'countries': COUNTRIES, 'target': TARGET, 'options': OPTIONS,
           'question': QUESTION, 'persona_fields': PERSONA_FIELDS, 'static_file': STATIC_FILE,
           'target_file': TARGET_FILE, 'sample_file': SAMPLE_PATH, 'seed': SEED, 'system_prompt': SYSTEM},
          open(f'{out}/config.json', 'w'), indent=2)
print('Saved to', out, os.listdir(out))

Saved to /content/drive/MyDrive/EIB_runs/six_country_static_baseline/run_20261027_0327_claude-haiku-5-5 ['summary.csv', 'per_country.csv', 'distribution.csv', 'llm_predictions.csv', 'config.json']
